# Expressions - Rust

All 15 Rust examples from [docs/expressions.md](https://platob.github.io/yggdryl/expressions/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::Expr;

// Parsed from SQL-like text, built with constructors, or built with the
// language's own operators - and all three are the same value.
let parsed: Expr = "venue = 'XNAS' AND price > 10".parse()?;
let built = Expr::column("venue")
    .eq(Expr::literal("XNAS"))
    .and(Expr::column("price").gt(Expr::literal(10)));

use yggdryl::expressions::{col, lit};
let with_operators = col("venue").eq(lit("XNAS")) & col("price").gt(lit(10));

assert_eq!(parsed, built);
assert_eq!(parsed, with_operators);

// The canonical text round-trips, and says which columns a read must decode.
assert_eq!(parsed.to_string(), "venue = 'XNAS' AND price > 10");
assert_eq!(parsed.columns(), vec!["venue".to_owned(), "price".to_owned()]);

## Take a filter from untrusted text

In [ ]:
use yggdryl::{Error, Expr};

// A byte offset, and what was expected there.
let error = "venue = ".parse::<Expr>().unwrap_err();
let Error::Parse { position, reason, .. } = &error else {
    panic!("expected a parse error, got {error}");
};
assert_eq!(*position, 8);
assert!(reason.contains("expected a value"));

// An unterminated delimiter reports the *opener*, which is the position a
// caller can actually fix.
let error = "venue = 'XNAS".parse::<Expr>().unwrap_err();
let Error::Parse { position, .. } = &error else { panic!("{error}") };
assert_eq!(*position, 8);

// And a function outside the closed vocabulary names the vocabulary it is
// not in, rather than being looked up somewhere.
let error = "system('rm -rf /') = 1".parse::<Expr>().unwrap_err();
assert!(error.to_string().contains("coalesce"));

## Name a column that needs quoting

In [ ]:
use yggdryl::Expr;

// Three spellings in, one spelling out.
for text in ["\"total amount\" = 1", "`total amount` = 1", "[total amount] = 1"] {
    assert_eq!(text.parse::<Expr>()?.to_string(), "\"total amount\" = 1");
}

// A doubled closer embeds the delimiter itself.
let embedded: Expr = "\"say \"\"hi\"\" now\" = 1".parse()?;
assert_eq!(embedded.columns(), vec!["say \"hi\" now".to_owned()]);

// Whitespace inside an encapsulator is data, and survives the round trip.
let padded: Expr = "\"  a  \" IS NULL".parse()?;
assert_eq!(padded.columns(), vec!["  a  ".to_owned()]);
assert_eq!(padded.to_string(), "\"  a  \" IS NULL");

In [ ]:
use yggdryl::Expr;

// A column compared to a string.
let column: Expr = "\"venue\" = 'XNAS'".parse()?;
assert!(!column.columns().is_empty());

// Two strings, which are not equal, so the whole thing folds to FALSE.
let strings: Expr = "'venue' = 'XNAS'".parse()?;
assert!(strings.columns().is_empty());
assert!(strings.simplify().is_always_false());

## Reach inside a value

In [ ]:
use yggdryl::{DataType, Expr, Value};

let schema = DataType::from_fields([
    DataType::list(DataType::Int64.nullable_field("item")).nullable_field("tags"),
    DataType::Utf8.nullable_field("path"),
])?
.required_field("row");
let row = Value::record(
    schema.data_type().clone(),
    [
        Value::from_sequence([Value::I64(10), Value::I64(20), Value::I64(30)]),
        Value::from("abcdef"),
    ],
)?;
let read = |text: &str| -> yggdryl::Result<Value> {
    text.parse::<Expr>()?.bind(&schema)?.evaluate(&row)
};

assert_eq!(read("tags[0]")?, Value::I64(10));      // 0-based
assert_eq!(read("tags[-1]")?, Value::I64(30));     // from the end
assert_eq!(read("tags[99]")?, Value::Null);        // out of range is null
assert_eq!(read("tags[1:3]")?.len(), 2);           // half-open
assert_eq!(read("tags[3:1]")?.len(), 0);           // inverted is empty
assert_eq!(read("path[1:3]")?, Value::from("bc")); // text slices characters

// A range is not BETWEEN: this one is a predicate over a scalar.
let predicate: Expr = "tags[0] BETWEEN 1 AND 3".parse()?;
assert_eq!(predicate.to_string(), "tags[0] BETWEEN 1 AND 3");

## Bind once, evaluate many

In [ ]:
use yggdryl::{DataType, Expr, Value};

let schema = DataType::from_fields([
    DataType::Decimal128 { precision: 10, scale: 2 }.nullable_field("price"),
])?
.required_field("row");

// The text literal became the column's own decimal, once, at bind time.
let bound = "price > '10.5'".parse::<Expr>()?.bind(&schema)?;
assert_eq!(bound.to_expr().to_string(), "price > 10.50");

// One plan, used over a row and over a batch - the same answer both ways.
let predicate = bound.into_predicate()?;
let row = Value::record(schema.data_type().clone(), [Value::Decimal(2_000, 2)])?;
assert!(predicate.matches(&row)?);

In [ ]:
use yggdryl::{DataType, Expr};

let schema = DataType::from_fields([
    DataType::Int64.nullable_field("price"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");

let error = "prise > 1".parse::<Expr>()?.bind(&schema).unwrap_err();
let message = error.to_string();
assert!(message.contains("prise"));
assert!(message.contains("price, venue"));

## Null semantics

In [ ]:
use yggdryl::{DataType, Expr, Value};

let schema = DataType::from_fields([DataType::Utf8.nullable_field("venue")])?
    .required_field("row");
let missing = Value::record(schema.data_type().clone(), [Value::Null])?;

let keep = |text: &str| -> yggdryl::Result<bool> {
    text.parse::<Expr>()?.bind(&schema)?.into_predicate()?.matches(&missing)
};

// `venue <> 'XNAS'` does NOT select the rows whose venue is null.
assert!(!keep("venue <> 'XNAS'")?);
// `venue IS NULL` is how that is asked.
assert!(keep("venue IS NULL")?);
// And so `DELETE WHERE price > 10` would leave a null price alone.
assert!(!keep("venue > 'A'")?);

## Select and compute columns

In [ ]:
use yggdryl::{DataType, Field, Selection};

let schema = DataType::from_fields([
    DataType::Int64.nullable_field("price"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");

let selection: Selection = "venue, price * 2 AS doubled, price".parse()?;

// The root the selection produces, with nothing opened and no data read.
use yggdryl::expressions::Apply;
let root = selection.apply_field(&schema)?;
let names: Vec<&str> = root.fields().iter().map(Field::name).collect();
assert_eq!(names, vec!["venue", "doubled", "price"]);

// An unnamed computed column takes its own canonical spelling.
let unnamed: Selection = "price * 2".parse()?;
assert_eq!(unnamed.apply_field(&schema)?.fields()[0].name(), "price * 2");

// The columns a read must decode are the columns the selection *reads*.
assert_eq!(selection.columns(), vec!["venue".to_owned(), "price".to_owned()]);

## Apply to what you already hold

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::expressions::{Apply, ArrowApply};
use yggdryl::{DataType, Expr, Value, arrow};

let schema = DataType::from_fields([
    DataType::Utf8.nullable_field("venue"),
    DataType::Int64.nullable_field("id"),
])?
.required_field("row");
let batch = RecordBatch::try_new(
    arrow::schema_from_field(&schema)?,
    vec![
        Arc::new(StringArray::from(vec![Some("XNAS"), Some("XNYS"), None])),
        Arc::new(Int64Array::from(vec![Some(1_i64), Some(2), Some(3)])),
    ],
)?;

let filter: Expr = "venue = 'XNAS'".parse()?;

// A batch, filtered.
let kept = filter.apply_arrow_batch(batch.clone())?;
assert_eq!(kept.num_rows(), 1);

// A stream, filtered lazily - the schema answers before the first batch.
let reader = arrow::batch_reader(batch.schema(), [batch.clone()]);
let streamed = filter.apply_arrow_batch_reader(reader)?;
assert_eq!(streamed.schema(), batch.schema());

// A schema alone: nothing opened, no data allocated. Filtering does not
// change a schema, so the answer is the root it was bound to.
assert_eq!(filter.apply_field(&schema)?.field_len(), 2);

// And one row at a time.
let row = Value::record(schema.data_type().clone(), [Value::from("XNAS"), Value::I64(1)])?;
assert_eq!(filter.apply_value(&schema, &row)?, Value::Bool(true));

## Watch the optimizer work

In [ ]:
use yggdryl::{DataType, Expr};

let schema = DataType::from_fields([DataType::Int32.nullable_field("id")])?
    .required_field("row");

// A long OR of equalities with a cast around the column: the worst shape a
// pushdown can meet, because a cast on a column destroys pruning outright.
let written: Expr =
    "CAST(id AS int64) = 1 OR CAST(id AS int64) = 2 OR CAST(id AS int64) = 3".parse()?;
let bound = written.bind(&schema)?;

// One IN list, and the cast moved to the literals - so the column is now
// comparable against statistics again.
assert_eq!(bound.to_expr().to_string(), "id IN (1, 2, 3)");

let explained = bound.explain();
assert!(explained.contains("cast moved from column to literal"));
assert!(explained.contains("OR of equalities to an IN list"));

In [ ]:
use yggdryl::{DataType, Expr};

// `a = a` is unknown when `a` is null, so it is never TRUE.
assert_eq!("a = a".parse::<Expr>()?.simplify().to_string(), "a = a");
assert_eq!("a <> a".parse::<Expr>()?.simplify().to_string(), "a <> a");

// A contradiction is unknown when the column is null, so it folds only
// where the schema proves it cannot be.
let nullable = DataType::from_fields([DataType::Int64.nullable_field("a")])?
    .required_field("row");
let held = "a > 5 AND a < 3".parse::<Expr>()?.bind(&nullable)?;
assert!(!held.is_always_false());
assert!(held.explain().contains("declined contradictory range"));

let required = DataType::from_fields([DataType::Int64.required_field("a")])?
    .required_field("row");
let folded = "a > 5 AND a < 3".parse::<Expr>()?.bind(&required)?;
assert!(folded.is_always_false());

## Prune without reading

In [ ]:
use yggdryl::expressions::{BoundColumn, Certainty, ColumnStats, StatsSource};
use yggdryl::{DataType, Expr, Value};

struct File;

impl StatsSource for File {
    fn stats(&self, column: &BoundColumn) -> Option<ColumnStats> {
        match column.name() {
            // Every row under `venue=XNAS/` holds that one value.
            "venue" => Some(ColumnStats::constant(Value::from("XNAS"))),
            "id" => Some(ColumnStats::range(Value::I64(100), Value::I64(200))),
            _ => None,
        }
    }
}

let schema = DataType::from_fields([
    DataType::Utf8.nullable_field("venue"),
    DataType::Int64.nullable_field("id"),
])?
.required_field("row");
let decide = |text: &str| -> yggdryl::Result<Certainty> {
    Ok(text.parse::<Expr>()?.bind(&schema)?.into_predicate()?.evaluate_stats(&File))
};

// Provably nothing: the file is never opened.
assert_eq!(decide("venue = 'XNYS'")?, Certainty::AlwaysFalse);
assert_eq!(decide("id > 500")?, Certainty::AlwaysFalse);
// Provably everything: the conjunct never runs against a single row.
assert_eq!(decide("venue = 'XNAS'")?, Certainty::AlwaysTrue);
// Unsettled: the rows have to answer.
assert_eq!(decide("id > 150")?, Certainty::Maybe);
// Nothing known: never prune.
assert_eq!(decide("absent > 1").is_err(), true);

In [ ]:
use yggdryl::expressions::{BoundColumn, ColumnStats, StatsSource};
use yggdryl::{DataType, Expr, Value};

struct Partition;

impl StatsSource for Partition {
    fn stats(&self, column: &BoundColumn) -> Option<ColumnStats> {
        (column.name() == "venue").then(|| ColumnStats::constant(Value::from("XNAS")))
    }
}

let schema = DataType::from_fields([
    DataType::Utf8.nullable_field("venue"),
    DataType::Int64.nullable_field("id"),
])?
.required_field("row");
let predicate = "venue = 'XNAS' AND id > 100"
    .parse::<Expr>()?
    .bind(&schema)?
    .into_predicate()?;

// The partition settled one conjunct, so only the other reaches the rows.
let residual = predicate.residual(&Partition).expect("the file can match");
assert_eq!(residual.len(), 1);
assert_eq!(residual[0].to_string(), "id > 100");

// An empty list would mean "every row matches"; `None` means "no row does".
struct Elsewhere;
impl StatsSource for Elsewhere {
    fn stats(&self, column: &BoundColumn) -> Option<ColumnStats> {
        (column.name() == "venue").then(|| ColumnStats::constant(Value::from("XNYS")))
    }
}
assert!(predicate.residual(&Elsewhere).is_none());

## Migration

In [ ]:
use yggdryl::io::partition::pairs_to_expr;

// The pair form builds exactly the expression, quoting included.
assert_eq!(
    pairs_to_expr(&[("venue", "XNAS"), ("total amount", "3")]).to_string(),
    "venue = 'XNAS' AND \"total amount\" = '3'"
);
// And the text `null` means the absence a directory name spells.
assert_eq!(pairs_to_expr(&[("venue", "null")]).to_string(), "venue IS NULL");